# Data yield report (browse → load → QC → pool)

Answer, across system conditions (**modular / rigid / mouse / turtle**):

1. **DeepLabCut yield** — pooled Pupil/edge likelihood histograms (same selection rule as the preprocessing GUI: newest `*filtered*` CSV, else newest).
2. **Ellipse yield** — % frames with fitted ellipses on `*raw_verified*` eye CSVs after in-memory application of `noise_epochs_*` + `manual_event_annotations.csv` (no disk writes).
3. **Missing-data structure** — of missing frames, what fraction sits in short gaps (≤5 frames) vs longer epochs; median missing-epoch length.

**Workflow.** Browse/tag once, then run **Load data** to pull all eye + DLC + removal catalogs into a RAM `DATA` object. QC / pool / finalize reuse `DATA` so network volumes are not re-read.

Figures mirror the jitter grouping: modular+rigid overlay; mouse alone; turtle alone.

Outputs land in `outputs/<run>/{figures,metadata}/`; empty `TAG` overwrites `yield_latest`.

## 0. Setup

In [1]:
%matplotlib inline
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "src" / "eye_tracking_system_tools").is_dir():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

from eye_tracking_system_tools.analysis.data_yield import (
    DEFAULT_LIKELIHOOD_THR,
    finalize_yield_export,
    load_yield_dataset,
    retag_dataset_block,
    seed_yield_registry_from_jitter,
    write_yield_registry,
)
from eye_tracking_system_tools.analysis.data_yield_gui import (
    YieldPoolSelector,
    yield_registry_table,
)
from eye_tracking_system_tools.analysis.jitter_epochs import read_registry_blocks
from eye_tracking_system_tools.analysis.jitter_gui import JitterBlockBrowser
from eye_tracking_system_tools.analysis.run_layout import resolve_run_dir

DATA = None  # filled by the Load data cell

REGISTRY = REPO / "configs" / "data_yield_blocks.yaml"
JITTER_REGISTRY = REPO / "configs" / "jitter_mount_blocks.yaml"
OUT_ROOT = REPO / "outputs"
TAG = ""  # empty → yield_latest (overwrite); e.g. "cohort_v1" → yield_cohort_v1
LIKELIHOOD_THR = DEFAULT_LIKELIHOOD_THR
N_BINS = 50

RUN = resolve_run_dir(OUT_ROOT, TAG or None, prefix="yield", default_name="yield_latest")
print("REPO    :", REPO)
print("registry:", REGISTRY)
print("run     :", RUN.run_dir)
print("  figures :", RUN.figures_dir)
print("  metadata:", RUN.metadata_dir)

REPO    : /Users/nimi/Projects/PETS
registry: /Users/nimi/Projects/PETS/configs/data_yield_blocks.yaml
run     : /Users/nimi/Projects/PETS/outputs/yield_latest
  figures : /Users/nimi/Projects/PETS/outputs/yield_latest/figures
  metadata: /Users/nimi/Projects/PETS/outputs/yield_latest/metadata


## 1. Collect / tag blocks

Reuses the jitter filesystem browser with `require="eye"` (eye CSV present) and mount tags including **turtle**.

Optional: seed from the jitter registry (same cohort) if the yield YAML is empty.

In [2]:
# Seed once from the jitter cohort (no-op if yield registry already has blocks).
seed_yield_registry_from_jitter(REGISTRY, JITTER_REGISTRY, overwrite=False)
print("registry blocks:", len(read_registry_blocks(REGISTRY)))

registry blocks: 49


In [3]:
browser = JitterBlockBrowser(
    REGISTRY,
    repo=REPO,
    registry_format="jitter",
    require="eye",
)
browser

In [4]:
browser.save()
SPECS = read_registry_blocks(REGISTRY)
print(f"{len(SPECS)} block(s) loaded")
for s in SPECS:
    print(f"  [{s.mount_type}] {s.block_key}")

49 block(s) loaded
  [rigid] PV_106_block_008
  [rigid] PV_106_block_009
  [rigid] PV_106_block_010
  [rigid] PV_106_block_011
  [rigid] PV_106_block_012
  [rigid] PV_106_block_013
  [rigid] PV_106_block_014
  [rigid] PV_106_block_015
  [rigid] PV_106_block_016
  [rigid] PV_126_block_006
  [rigid] PV_126_block_007
  [rigid] PV_126_block_008
  [rigid] PV_126_block_009
  [rigid] PV_126_block_010
  [rigid] PV_126_block_011
  [rigid] PV_126_block_012
  [rigid] PV_126_block_013
  [rigid] PV_126_block_014
  [rigid] PV_126_block_015
  [rigid] PV_143_block_001
  [rigid] PV_143_block_002
  [rigid] PV_143_block_003
  [rigid] PV_143_block_004
  [modular] PV_24_block_012
  [modular] PV_24_block_013
  [modular] PV_24_block_034
  [modular] PV_62_block_023
  [modular] PV_62_block_024
  [modular] PV_62_block_025
  [modular] PV_62_block_026
  [modular] PV_62_block_027
  [mouse] M_002_block_012
  [mouse] M_002_block_013
  [mouse] M_002_block_014
  [mouse] M_002_block_015
  [rigid] PV_228_block_001
  [ri

## 2. Load data (once)

Pull eye CSVs, DLC likelihoods, and removal catalogs from the network into RAM.
This is the slow step — expect several minutes for a full cohort on `/Volumes/...`.
Everything after this reuses `DATA` and should be fast.

In [5]:
SPECS = read_registry_blocks(REGISTRY)
DATA = load_yield_dataset(SPECS, verbose=True)
DATA.summarize()

[1/49] loading PV_106_block_008 [rigid] …
  eyes=2/2  likelihood_samples=1183070
[2/49] loading PV_106_block_009 [rigid] …
  eyes=2/2  likelihood_samples=1178700
[3/49] loading PV_106_block_010 [rigid] …
  eyes=2/2  likelihood_samples=1167170
[4/49] loading PV_106_block_011 [rigid] …
  eyes=2/2  likelihood_samples=2665500
[5/49] loading PV_106_block_012 [rigid] …
  eyes=2/2  likelihood_samples=2301090
[6/49] loading PV_106_block_013 [rigid] …
  eyes=2/2  likelihood_samples=981480
[7/49] loading PV_106_block_014 [rigid] …
  eyes=2/2  likelihood_samples=780530
[8/49] loading PV_106_block_015 [rigid] …
  eyes=2/2  likelihood_samples=395010
[9/49] loading PV_106_block_016 [rigid] …
  eyes=2/2  likelihood_samples=1417720
[10/49] loading PV_126_block_006 [rigid] …
  eyes=2/2  likelihood_samples=1584410
[11/49] loading PV_126_block_007 [rigid] …
  eyes=2/2  likelihood_samples=2437060
[12/49] loading PV_126_block_008 [rigid] …
  eyes=2/2  likelihood_samples=3993170
[13/49] loading PV_126_block

{'n_blocks': 49,
 'n_eyes_loaded': 96,
 'n_eye_rows': 7557646,
 'n_likelihood_samples': 79209584,
 'n_errors': 0,
 'loaded_utc': '2026-08-06T18:27:07.318136+00:00'}

## 3. QC

Uses the in-memory `DATA` cache — should be near-instant. Shows paths, removal catalogs present, and post-mask yield %.

In [6]:
assert DATA is not None, "Run the Load data cell first"
qc = yield_registry_table(SPECS, dataset=DATA, likelihood_threshold=LIKELIHOOD_THR)
qc

,animal,block,mount_type,dlc_left,dlc_right,eye_left,eye_left_rule,eye_right,eye_right_rule,noise_left,noise_right,manual_annotations,yield_left_pct,yield_right_pct,likelihood_left_n,likelihood_right_n,block_path
0,PV_106,block_008,rigid,pv_106_d3_t1DLC_resnet_50_Eye_Tracking_pipline...,pv_106_d3_t1DLC_resnet_50_Eye_Tracking_pipline...,left_eye_data_degrees_raw_verified.csv,raw_verified,right_eye_data_degrees_raw_verified.csv,raw_verified,False,False,False,90.40,92.79,590740,592330,/Volumes/Data-2/Nimrod/experiments/PV_106/2025...
1,PV_106,block_009,rigid,pv_106_d3_t2DLC_resnet_50_Eye_Tracking_pipline...,pv_106_d3_t2DLC_resnet_50_Eye_Tracking_pipline...,left_eye_data_degrees_raw_verified.csv,raw_verified,right_eye_data_degrees_raw_verified.csv,raw_verified,False,False,False,85.97,88.33,588430,590270,/Volumes/Data-2/Nimrod/experiments/PV_106/2025...
2,PV_106,block_010,rigid,pv_106_d3_t3DLC_resnet_50_Eye_Tracking_pipline...,pv_106_d3_t3DLC_resnet_50_Eye_Tracking_pipline...,left_eye_data_degrees_raw_verified.csv,raw_verified,right_eye_data_degrees_raw_verified.csv,raw_verified,False,False,False,83.08,81.93,582780,584390,/Volumes/Data-2/Nimrod/experiments/PV_106/2025...
3,PV_106,block_011,rigid,pv_106_d3_t4DLC_resnet_50_Eye_Tracking_pipline...,pv_106_d3_t4DLC_resnet_50_Eye_Tracking_pipline...,left_eye_data_degrees_raw_verified.csv,raw_verified,right_eye_data_degrees_raw_verified.csv,raw_verified,False,False,False,83.91,91.27,1332690,1332810,/Volumes/Data-2/Nimrod/experiments/PV_106/2025...
4,PV_106,block_012,rigid,pv_106_d3_t5DLC_resnet_50_Eye_Tracking_pipline...,pv_106_d3_t5DLC_resnet_50_Eye_Tracking_pipline...,left_eye_data_degrees_raw_verified.csv,raw_verified,right_eye_data_degrees_raw_verified.csv,raw_verified,False,False,False,71.84,82.74,1149140,1151950,/Volumes/Data-2/Nimrod/experiments/PV_106/2025...
5,PV_106,block_013,rigid,pv_106_d3_t6DLC_resnet_50_Eye_Tracking_pipline...,pv_106_d3_t6DLC_resnet_50_Eye_Tracking_pipline...,left_eye_data.csv,newest,right_eye_data.csv,newest,False,False,False,96.43,98.43,588490,392990,/Volumes/Data-2/Nimrod/experiments/PV_106/2025...
6,PV_106,block_014,rigid,imu_trial2DLC_resnet_50_Eye_Tracking_piplineMa...,imu_trial2DLC_resnet_50_Eye_Tracking_piplineMa...,left_eye_data_degrees_raw_verified.csv,raw_verified,right_eye_data_degrees_raw_verified.csv,raw_verified,False,False,False,91.00,90.25,391180,389350,/Volumes/Data-2/Nimrod/experiments/PV_106/2025...
7,PV_106,block_015,rigid,imu_trial4_preyDLC_resnet_50_Eye_Tracking_pipl...,imu_trial4_preyDLC_resnet_50_Eye_Tracking_pipl...,left_eye_data_degrees_raw_verified.csv,raw_verified,right_eye_data_degrees_raw_verified.csv,raw_verified,False,False,False,89.56,89.78,198000,197010,/Volumes/Data-2/Nimrod/experiments/PV_106/2025...
8,PV_106,block_016,rigid,imu_trial5_preyDLC_resnet_50_Eye_Tracking_pipl...,imu_trial5_preyDLC_resnet_50_Eye_Tracking_pipl...,left_eye_data_degrees_raw_verified.csv,raw_verified,right_eye_data_degrees_raw_verified.csv,raw_verified,False,False,False,97.53,97.42,710170,707550,/Volumes/Data-2/Nimrod/experiments/PV_106/2025...
9,PV_126,block_006,rigid,hunter7_LEDLC_resnet_50_Eye_Tracking_piplineMa...,hunter7DLC_resnet_50_Eye_Tracking_piplineMar1s...,left_eye_data.csv,newest,right_eye_data.csv,newest,False,False,False,98.69,98.31,792240,792170,/Volumes/Data-2/Nimrod/experiments/PV_126/2024...


## 4. Pool + plot

Tick blocks to include, then run. Reuses `DATA` (no re-read from the server). Writes:

- `figures/yield_dlc_likelihood_{modular_vs_rigid,mouse,turtle}.pdf`
- `figures/yield_ellipse_{modular_vs_rigid,mouse,turtle}.pdf` (value labels + metric footnote)
- `figures/yield_ellipse_by_condition.pdf` (yield % only: modular / rigid / mouse / turtle)
- `metadata/per_block_yield.csv`, `metadata/yield_pool_summary.yaml`, `metadata/data_yield_report.pickle`

**Re-tag without reloading:** e.g. `retag_dataset_block(DATA, "PV_24_block_001", "rigid")` then `write_yield_registry(REGISTRY, DATA.specs)` and re-run pool.

In [ ]:
assert DATA is not None, "Run the Load data cell first"
pool = YieldPoolSelector(
    SPECS,
    RUN.figures_dir,
    RUN.metadata_dir,
    dataset=DATA,
    likelihood_threshold=LIKELIHOOD_THR,
    n_bins=N_BINS,
)
pool

## 5. Finalize

Dated export under `outputs/yield_report_<tag>_<YYYYmmdd>_<HH>_<MM>/` (also uses `DATA`).

In [ ]:
assert DATA is not None, "Run the Load data cell first"
EXPORT_TAG = TAG or "latest"
EXPORT_NOTES = "Data yield: DLC likelihood + raw_verified ellipse completeness after removals"

result = finalize_yield_export(
    SPECS,
    RUN.metadata_dir,
    OUT_ROOT,
    tag=EXPORT_TAG,
    include=pool.selected,
    likelihood_threshold=float(pool.thr.value),
    n_bins=int(pool.n_bins.value),
    notes=EXPORT_NOTES,
    dataset=DATA,
)
print(result)

## Notes

| Artifact | Meaning |
|----------|---------|
| `DATA` | RAM cache from §2 — eye frames (slim cols), DLC likelihoods, noise epochs, manual intervals |
| DLC CSV | `eye_videos/{LE,RE}/…` newest `*filtered*` (else newest) via `dlc_csv_io` |
| Eye CSV | Prefer `*raw_verified*`; else newest `left/right_eye_data*.csv` |
| Removals | In-memory only: all `noise_epochs_*` categories + flagged `manual_event_annotations.csv` |
| Fitted | Finite `center_x` |
| Short gap | Contiguous missing run ≤ 5 frames |

Turtle blocks can be added later with the browser (`mount_type: turtle`); empty turtle pools simply skip those figures.

Re-run §2 after changing the registry; QC / pool / finalize do not re-hit the server when `DATA` is passed.